## Φάση 0 — Ποιος κριτής φορτώνει;

Το `llm_client` περνάει `trust_remote_code=True`, οπότε ένα μοντέλο που κουβαλάει
δικό του modeling file θα τρέξει εκείνο -- και του Phi-3.5-mini είναι γραμμένο για
παλιότερο transformers, οπότε σκάει στην πρώτη generate με
`DynamicCache has no attribute from_legacy_cache`. Τα πέντε μοντέλα του corpus
χρησιμοποιούν native αρχιτεκτονικές, γι' αυτό δεν το συνάντησαν ποτέ.

Αντί να μαντεύουμε, δοκιμάζουμε: 3 μηνύματα ανά υποψήφιο, δευτερόλεπτα ο καθένας.
Ο πρώτος που περνάει είναι ο κριτής.

In [ ]:
import subprocess, sys

CANDIDATES = [
    'HuggingFaceTB/SmolLM2-1.7B-Instruct',   # from scratch by HF, native, 1.7B
    'ibm-granite/granite-3.1-2b-instruct',    # IBM, native, 2B
    'stabilityai/stablelm-2-1_6b-chat',       # Stability, native, 1.6B
]

JUDGE_MODEL = None
for m in CANDIDATES:
    print('=' * 60)
    print('δοκιμή:', m)
    r = subprocess.run([sys.executable, 'llm_judge.py', 'label',
                        '--roots', *ROOTS, '--games', 'pd',
                        '--from-csv', HUMAN[0], '--limit', '3',
                        '--judge-model', m, '--out-dir', '/kaggle/working/probe'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        JUDGE_MODEL = m
        print('OK ->', m)
        break
    tail = (r.stderr or r.stdout).strip().split('
')[-1]
    print('απέτυχε:', tail[:160])

assert JUDGE_MODEL, 'Κανένας υποψήφιος δεν φόρτωσε -- δες τα μηνύματα παραπάνω.'
print('
ΚΡΙΤΗΣ:', JUDGE_MODEL)


# LLM-as-judge — μέτρηση εξαπάτησης

**Τι μετράμε.** Ένα μήνυμα που σηματοδοτεί πρόθεση συνεργασίας, και ο ίδιος ο
αποστολέας αποστατεί τον ίδιο γύρο. Ο κριτής βλέπει **μόνο το κείμενο** — ποτέ
την ενέργεια — ώστε η κρίση του για την πρόθεση να μην μολύνεται από την έκβαση.

**Γιατί κριτής και όχι λίστα λέξεων.** Η λεξιλογική μέτρηση μετράει κάθε μήνυμα
που περιέχει συνεργατική λέξη, και υπερμετράει χονδρικά: μια υπόθεση («if we
both cooperate we would each gain 4») και μια ερώτηση («are you both committed
to cooperating?») δεν είναι δεσμεύσεις. Απαιτώντας υπόσχεση πρώτου προσώπου, τα
αυθόρμητα αθετημένα «ψέματα» έπεσαν από 893 σε 461 — και το σενάριο
`counterfactual` είναι μολυσμένο εξ ορισμού, αφού η οδηγία του **επιβάλλει**
δομή IF/WOULD.

**Ο κριτής.** `microsoft/Phi-3.5-mini-instruct`, 3,8B, τοπικά. Εκτός και από
τις τρεις οικογένειες του corpus (Qwen, Llama, Gemma), άρα χωρίς self-bias, και
χωρίς κανένα API key — το αποτέλεσμα αναπαράγεται από το repo και μόνο.

**Το μέγεθος του κριτή δεν χρειάζεται να ταιριάζει με των παικτών.** Ο κριτής
είναι όργανο, όχι υποκείμενο· η απαίτηση είναι ακρίβεια, και η ακρίβεια
μετριέται. Αυτό κάνει η Φάση 1.

## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add Data → ανέβασε ως Kaggle Dataset:
   - τους **δέκα φακέλους** των runs (`<model>_star`, `<model>_cycle`)
   - το `human_labels.csv` σου, αφού συμπληρώσεις τη στήλη
     `human_is_coop_signal` στο `human_labels_TEMPLATE.csv`

Το Phi-3.5-mini δεν είναι gated, οπότε `HF_TOKEN` δεν χρειάζεται.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

head = subprocess.run(['git', 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()
# Το repo root ΕΙΝΑΙ το package -- agent.py, llm_judge.py κλπ κάθονται στην
# κορυφή του clone. Αν αυτό αλλάξει, να αποτύχει εδώ με σαφές μήνυμα και
# όχι δέκα κελιά παρακάτω με ImportError.
assert os.path.exists('llm_judge.py'), (
    f'Το llm_judge.py δεν είναι στο {os.getcwd()} -- άλλαξε η δομή του repo;')
print('cwd :', os.getcwd())
print('HEAD:', head)

# Gate: το --from-csv είναι αυτό που κάνει την επικύρωση να μετράει το ίδιο
# σύνολο που χαρακτηρίστηκε στο χέρι. Χωρίς αυτό το notebook δεν έχει νόημα.
assert '--from-csv' in open('llm_judge.py').read(), \
    'Το repo δεν έχει το --from-csv -- κάνε git push πρώτα.'
print('gate ok')

## Πού είναι τα δεδομένα

Τα run JSON δεν είναι στο repo (είναι gitignored). Το κελί ψάχνει στο
`/kaggle/input` για τους δέκα φακέλους και σταματάει με σαφές μήνυμα αν λείπουν,
αντί να τρέξει σε άδειο σύνολο και να βγάλει μηδενικά.

In [ ]:
import glob, os

MODELS = ['Llama-3.1-8B-Instruct', 'Qwen2.5-7B-Instruct', 'Qwen3-4B',
          'gemma-2-2b-it', 'gemma-2-9b-it']
WANT = [f'{m}_{t}' for m in MODELS for t in ('star', 'cycle')]

ROOTS = []
for name in WANT:
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True)
            if os.path.isdir(h)]
    if hits:
        ROOTS.append(hits[0])

missing = set(WANT) - {os.path.basename(r) for r in ROOTS}
if missing:
    raise SystemExit(
        f'Λείπουν {len(missing)} φάκελοι runs: {sorted(missing)}
'
        'Ανέβασέ τους ως Kaggle Dataset (Add Data) και ξανατρέξε το κελί.')

n_json = sum(len(glob.glob(os.path.join(r, '**', '*.json'), recursive=True))
             for r in ROOTS)
print(f'{len(ROOTS)} φάκελοι, {n_json} json')

# Ανεκτικό pattern: το Excel σώζει συχνά ως human_labels.csv.csv, και ένα glob
# που ζητάει ακριβές όνομα αποτυγχάνει ενώ το αρχείο είναι ανεβασμένο. Το
# TEMPLATE εξαιρείται -- είναι το ασυμπλήρωτο.
HUMAN = [f for f in glob.glob('/kaggle/input/**/human_labels*.csv', recursive=True)
         if 'TEMPLATE' not in os.path.basename(f)]
if HUMAN:
    print('human_labels:', HUMAN[0])
else:
    # Μη σε αφήνει σε αδιέξοδο: δείξε τι CSV υπάρχουν όντως εκεί.
    seen = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    print('human_labels: ΔΕΝ ΒΡΕΘΗΚΕ')
    print('CSV που βλέπω στο /kaggle/input:')
    for f in seen[:20]:
        print('   ', f)
    if not seen:
        print('    (κανένα -- λείπει το Add Data ή το αρχείο δεν ανέβηκε)')

OUT = '/kaggle/working/judge'
os.makedirs(OUT, exist_ok=True)


## Φάση 1 — Επικύρωση του κριτή

Κρίνει **ακριβώς** τα ~117 μηνύματα που χαρακτήρισες στο χέρι, ώστε η
βαθμολόγηση να γίνει στο ίδιο σύνολο. Λίγα λεπτά.

In [ ]:
import subprocess, sys
assert HUMAN, 'Ανέβασε πρώτα το human_labels.csv'

subprocess.run([sys.executable, 'llm_judge.py', 'label',
                '--roots', *ROOTS,
                '--games', 'pd',
                '--from-csv', HUMAN[0],
                '--judge-model', JUDGE_MODEL,
                '--out-dir', OUT], check=True)

In [ ]:
import glob, subprocess, sys

labels = sorted(glob.glob(f'{OUT}/judge_labels_*.csv'))
print('labels:', labels)

subprocess.run([sys.executable, 'llm_judge.py', 'validate',
                '--human-labels', HUMAN[0],
                '--judge-labels', labels[-1]], check=True)

### Πώς διαβάζεται

| accuracy | τι κάνεις |
|---|---|
| **> 90 %** | προχώρα στη Φάση 2 με αυτόν τον κριτή |
| **80–90 %** | κοίτα το confusion. Χαμηλό *recall* = χάνει υποσχέσεις· χαμηλό *precision* = βλέπει υποσχέσεις παντού |
| **< 80 %** | άλλαξε κριτή στο κελί της Φάσης 2 (`--judge-model`) και ξανακάνε τη Φάση 1 |

Το νούμερο μπαίνει στη μεθοδολογία. Μέτρηση εξαπάτησης χωρίς επικύρωση έναντι
ανθρώπου δεν αντέχει στην εξέταση· με πίνακα σύγχυσης σε 117 χαρακτηρισμένα,
αντέχει.

## Φάση 2 — Όλο το corpus

**Μόνο αν η Φάση 1 πέρασε.** 25.600 μηνύματα, 16.394 μοναδικά κείμενα — η cache
κλειδώνει σε `judge_model||game||message`, οπότε ό,τι κρίθηκε στη Φάση 1 δεν
ξανατρέχει. Περίπου 1,5–2,5 ώρες σε T4.

Για άλλον κριτή, πρόσθεσε `'--judge-model', '<hf id>'`. Τα αρχεία εξόδου
κουβαλάνε το όνομα του μοντέλου, οπότε δύο κριτές δεν πατάει ο ένας τον άλλον.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, 'llm_judge.py', 'label',
                '--roots', *ROOTS,
                '--games', 'pd',
                '--judge-model', JUDGE_MODEL,
                '--out-dir', OUT], check=True)

In [ ]:
import shutil, os
zip_base = '/kaggle/working/judge_output'
shutil.make_archive(zip_base, 'zip', OUT)
print('κατέβασε αυτό:', zip_base + '.zip',
      f'({os.path.getsize(zip_base + ".zip")/1e6:.1f} MB)')
for f in sorted(os.listdir(OUT)):
    print(' ', f)

## Μετά

Κατέβασε το `judge_output.zip` στο `diplomatikh/cross_model_output_final/`. Μέσα:

- `judge_labels_<model>.csv` — μία γραμμή ανά μήνυμα, με την ετυμηγορία, την
  εμπιστοσύνη και μια πρόταση αιτιολόγησης
- `judge_deception_rate_<model>.csv` — ποσοστό ανά (μοντέλο, τοπολογία, κελί,
  παιχνίδι)
- `judge_cache_<model>.jsonl` — κράτα το, κάνει κάθε επανάληψη δωρεάν

Η σύγκριση που μας ενδιαφέρει είναι **κριτής έναντι λέξεων** ανά κελί: αν ο
κριτής ρίξει το `counterfactual` πολύ χαμηλά και κρατήσει ψηλά το
`framing_team`, επιβεβαιώνεται ότι το λεξιλόγιο μετρούσε υποθέσεις αντί για
δεσμεύσεις.